# LLM Inference Optimization: Speed, Memory, and Efficiency

Large Language Models are computationally expensive at inference time. A 7B parameter model in FP16 requires ~14 GB of GPU memory just for weights before accounting for activations, KV cache, and batch state. Serving these models efficiently at scale requires a layered set of optimizations spanning memory management, attention computation, decoding algorithms, quantization, and batching strategies.

This notebook surveys the key techniques that power modern LLM inference stacks (vLLM, TGI, TensorRT-LLM, llama.cpp), explaining the intuition, the mathematics, and the implementation patterns behind each approach.

## Optimization Landscape

| Technique | Type | Key Benefit | Primary Tools |
|---|---|---|---|
| KV Cache | Memory | Avoid recomputing past tokens | All transformers |
| Continuous Batching | Speed | Higher GPU utilization | vLLM, TGI |
| PagedAttention | Memory | Near-zero KV fragmentation | vLLM |
| FlashAttention | Both | $O(N)$ memory, faster kernels | Flash-Attn, xFormers |
| Speculative Decoding | Speed | Multiple tokens per step | vLLM, Medusa |
| Prefix/Prompt Caching | Speed | Reuse repeated prefixes | vLLM, Anthropic API |
| INT8 Quantization | Memory | 2x memory reduction | bitsandbytes |
| INT4 Quantization | Memory | 4x memory reduction | GPTQ, AWQ, GGUF |
| Tensor Parallelism | Speed | Multi-GPU scaling | TGI, TensorRT-LLM |
| SmoothQuant | Memory | Quantize activations safely | SmoothQuant |

> **Reading guide**: Each section is self-contained. Sections 1-3 cover the memory/compute foundation; sections 4-6 cover algorithmic speedups; sections 7-9 cover quantization; section 10 covers serving frameworks.


## KV Cache: The Memory Foundation

Transformer self-attention computes three projections for each token at each layer: **Query (Q)**, **Key (K)**, and **Value (V)**. During autoregressive generation, when producing token $t$, we compute attention over all previous tokens $1 \ldots t-1$ plus the current token $t$. Without caching, this requires recomputing K and V for every past token on every forward pass an $O(N^2)$ cost.

**KV Cache** stores the K and V projections for all past tokens so they don't need to be recomputed. Only the current token's Q, K, V need to be computed fresh; the stored K and V matrices are retrieved for the attention computation.

### Memory Formula

$$\text{KV Cache Size} = 2 \cdot n_{\text{layers}} \cdot n_{\text{heads}} \cdot d_{\text{head}} \cdot s \cdot b \cdot \text{dtype\_bytes}$$

where:
- $n_{\text{layers}}$ number of transformer layers
- $n_{\text{heads}}$ number of attention heads
- $d_{\text{head}}$ dimension per head ($d_{\text{model}} / n_{\text{heads}}$)
- $s$ sequence length (number of tokens)
- $b$ batch size
- $\text{dtype\_bytes}$ 2 for FP16/BF16, 4 for FP32, 1 for INT8
- The factor of 2 accounts for both K and V tensors

### Worked Example: LLaMA-7B

LLaMA-7B has: 32 layers, 32 heads, $d_{\text{head}} = 128$ (since $d_{\text{model}} = 4096$), FP16 (2 bytes).

$$\text{KV Cache} = 2 \times 32 \times 32 \times 128 \times 2048 \times 1 \times 2 \text{ bytes}$$
$$= 2 \times 32 \times 32 \times 128 \times 2048 \times 2 = 1{,}073{,}741{,}824 \text{ bytes} \approx 1 \text{ GB}$$

This is per sequence. With batch size 32, the KV cache alone consumes ~32 GB comparable to or exceeding the model weights themselves.

### Linear Growth with Sequence Length

KV cache grows **linearly** with sequence length $s$. Long-context models (128K tokens) face severe memory pressure:
- At 128K tokens, LLaMA-7B KV cache ≈ 64 GB (single batch) requires multi-GPU even for a 7B model.
- This is why techniques like **Grouped Query Attention (GQA)** and **Multi-Query Attention (MQA)** reduce $n_{\text{heads}}$ only in the K/V projections (not Q), dramatically shrinking KV cache.

With GQA ($n_{\text{kv\_heads}} < n_{\text{heads}}$), the formula becomes:

$$\text{KV Cache (GQA)} = 2 \cdot n_{\text{layers}} \cdot n_{\text{kv\_heads}} \cdot d_{\text{head}} \cdot s \cdot b \cdot \text{dtype\_bytes}$$

LLaMA-3-8B uses GQA with 8 KV heads vs 32 query heads an 4x KV cache reduction.


In [1]:
def compute_kv_cache_memory(
    n_layers: int,
    n_heads: int,
    d_head: int,
    seq_len: int,
    batch_size: int = 1,
    dtype_bytes: int = 2,  # FP16
    n_kv_heads: int = None,  # For GQA/MQA; defaults to n_heads
) -> float:
    """Compute KV cache memory in bytes."""
    if n_kv_heads is None:
        n_kv_heads = n_heads
    total = 2 * n_layers * n_kv_heads * d_head * seq_len * batch_size * dtype_bytes
    return total


def bytes_to_gb(b: float) -> float:
    return b / (1024 ** 3)


# Popular model configurations
# (name, n_layers, n_heads, d_model, n_kv_heads)
models = [
    ("LLaMA-7B",  32, 32, 4096, 32),
    ("LLaMA-13B", 40, 40, 5120, 40),
    ("LLaMA-70B", 80, 64, 8192,  8),  # GQA: 8 KV heads
    ("Mistral-7B",32, 32, 4096,  8),  # GQA: 8 KV heads
    ("LLaMA-3-8B",32, 32, 4096,  8),  # GQA: 8 KV heads
]

seq_lengths = [512, 2048, 8192, 32768, 131072]

# ---- Print header ----
col_w = 12
print(f"{'Model':<15}", end="")
for s in seq_lengths:
    label = f"{s//1024}K" if s >= 1024 else str(s)
    print(f"{label:>{col_w}}", end="")
print()
print("-" * (15 + col_w * len(seq_lengths)))

for (name, n_layers, n_heads, d_model, n_kv_heads) in models:
    d_head = d_model // n_heads
    print(f"{name:<15}", end="")
    for s in seq_lengths:
        mem = compute_kv_cache_memory(
            n_layers=n_layers,
            n_heads=n_heads,
            d_head=d_head,
            seq_len=s,
            batch_size=1,
            dtype_bytes=2,
            n_kv_heads=n_kv_heads,
        )
        gb = bytes_to_gb(mem)
        print(f"{gb:>{col_w}.3f}", end="")
    print()

print()
print("KV cache size in GB (FP16, batch=1)")
print("Seq lengths:", [f"{s//1024}K" if s >= 1024 else str(s) for s in seq_lengths])

# Show effect of GQA on LLaMA-70B
print("\n=== GQA Impact on LLaMA-70B (2K seq, batch=1) ===")
mha_mem = compute_kv_cache_memory(80, 64, 128, 2048, 1, 2, n_kv_heads=64)
gqa_mem = compute_kv_cache_memory(80, 64, 128, 2048, 1, 2, n_kv_heads=8)
print(f"  MHA (64 KV heads): {bytes_to_gb(mha_mem):.3f} GB")
print(f"  GQA ( 8 KV heads): {bytes_to_gb(gqa_mem):.3f} GB")
print(f"  Reduction: {mha_mem / gqa_mem:.1f}x")


Model                   512          2K          8K         32K        128K
---------------------------------------------------------------------------
LLaMA-7B              0.250       1.000       4.000      16.000      64.000
LLaMA-13B             0.391       1.562       6.250      25.000     100.000
LLaMA-70B             0.156       0.625       2.500      10.000      40.000
Mistral-7B            0.062       0.250       1.000       4.000      16.000
LLaMA-3-8B            0.062       0.250       1.000       4.000      16.000

KV cache size in GB (FP16, batch=1)
Seq lengths: ['512', '2K', '8K', '32K', '128K']

=== GQA Impact on LLaMA-70B (2K seq, batch=1) ===
  MHA (64 KV heads): 5.000 GB
  GQA ( 8 KV heads): 0.625 GB
  Reduction: 8.0x


## Batched Inference: Static vs Dynamic

### Static Batching

The simplest batching strategy: collect $B$ requests, pad all inputs to the maximum length in the batch, run a single forward pass. This is straightforward to implement but has two major inefficiencies:

1. **Padding waste**: If one request is 10 tokens and another is 2000 tokens, the first request wastes 1990 padded positions per layer on every forward pass.
2. **Head-of-line blocking**: The batch cannot return until the **longest** sequence finishes generation. Shorter sequences are idle while waiting.

GPU utilization under static batching is typically 20-40% in production workloads due to these inefficiencies.

### Dynamic / Continuous Batching (Orca)

The **Orca** paper (Yu et al., 2022) introduced *iteration-level scheduling*: rather than scheduling at the request granularity, schedule at the **iteration** (forward pass) granularity.

- At each iteration, the scheduler decides which requests participate.
- A request that has finished can be immediately evicted from the batch, and a new request can join.
- This eliminates head-of-line blocking: fast requests leave early, freeing capacity for new arrivals.

**Continuous batching** is now standard in all production serving frameworks (vLLM, TGI, TensorRT-LLM).

### Throughput vs Latency Tradeoff

| Strategy | Throughput | Latency | Use Case |
|---|---|---|---|
| Batch size = 1 | Low | Minimal | Interactive chat |
| Static large batch | High | High (due to padding & blocking) | Offline processing |
| Continuous batching | High | Low | Production serving |
| Chunked prefill | High | Controlled | Mixed workloads |

**Chunked prefill** (used in vLLM) splits long prompt processing into chunks, interleaving prefill and decode phases to bound latency for new requests even when long prompts are being processed.

### Prefill vs Decode Phases

LLM generation has two computationally distinct phases:

- **Prefill**: process the entire input prompt in parallel (one forward pass for all prompt tokens). This is compute-bound.
- **Decode**: autoregressively generate one token at a time. Each step is memory-bandwidth bound (the GPU reads all weights to generate one token).

Modern serving disaggregates these phases onto different hardware (prefill on compute-heavy GPUs, decode on memory-bandwidth-optimized GPUs) this is called **prefill-decode disaggregation** or **disaggregated inference**.


## PagedAttention: Virtual Memory for KV Cache

### The Fragmentation Problem

Traditional KV cache management allocates a contiguous memory block for each sequence, sized to the **maximum possible** output length. This causes two types of fragmentation:

- **Internal fragmentation**: Memory is pre-reserved for the maximum sequence length, but most sequences are much shorter. A request that generates 100 tokens wastes the memory reserved for the remaining 900 (if max is 1000).
- **External fragmentation**: As requests complete and new ones arrive, the memory pool develops gaps between allocations that are too small for new requests.

In practice, only **20-40%** of reserved KV cache memory is actually used in traditional systems.

### PagedAttention Solution

**PagedAttention** (vLLM, Kwon et al., 2023) directly borrows the OS virtual memory concept:

- KV cache is divided into fixed-size **blocks** (pages), each holding KV vectors for $B$ tokens.
- A **block table** maps logical token positions to physical block addresses like a page table.
- Physical blocks are allocated on demand as the sequence grows; they need not be contiguous.
- Completed sequences return their blocks to the free pool immediately.

```
Sequence A:  [Block 0] -> [Block 3] -> [Block 7]   (non-contiguous)
Sequence B:  [Block 1] -> [Block 5]                (different physical blocks)
Free blocks: [Block 2, Block 4, Block 6, ...]
```

### Benefits

| Property | Contiguous KV Cache | PagedAttention |
|---|---|---|
| Internal fragmentation | High (reserved but unused) | Near-zero (block granularity) |
| External fragmentation | High (gaps) | Zero (fixed-size blocks) |
| Memory utilization | 20-40% | >96% |
| Copy-on-write sharing | Hard | Native (beam search, parallel sampling) |

### Copy-on-Write for Beam Search

When doing beam search, multiple hypotheses share a common prefix. PagedAttention implements **copy-on-write** semantics: shared blocks are reference-counted. A block is only physically copied when one beam diverges (writes a new token), not for reads. This makes beam search memory-efficient without special-casing.

The CUDA kernel for PagedAttention must dereference the block table indirection at each attention step a small overhead compensated by much better memory utilization and throughput.


In [2]:
import math
from typing import Dict, List, Optional


class PagedKVCacheManager:
    """
    Simplified PagedAttention block manager.
    Demonstrates the block table concept for KV cache management.
    """

    def __init__(self, total_gpu_memory_gb: float, block_size: int, n_layers: int,
                 n_kv_heads: int, d_head: int, dtype_bytes: int = 2):
        self.block_size = block_size          # tokens per block
        self.n_layers = n_layers
        self.n_kv_heads = n_kv_heads
        self.d_head = d_head
        self.dtype_bytes = dtype_bytes

        # Memory per block (bytes): K + V tensors
        self.bytes_per_block = (
            2 * n_layers * n_kv_heads * d_head * block_size * dtype_bytes
        )

        total_bytes = int(total_gpu_memory_gb * (1024 ** 3))
        self.num_blocks = total_bytes // self.bytes_per_block

        # Free list of block IDs
        self.free_blocks: List[int] = list(range(self.num_blocks))

        # block_table[seq_id] = list of physical block IDs
        self.block_table: Dict[int, List[int]] = {}

        # Reference counts for copy-on-write
        self.ref_counts: Dict[int, int] = {bid: 0 for bid in range(self.num_blocks)}

    def _alloc_block(self) -> int:
        if not self.free_blocks:
            raise MemoryError("Out of KV cache blocks!")
        block_id = self.free_blocks.pop()
        self.ref_counts[block_id] = 1
        return block_id

    def _free_block(self, block_id: int):
        self.ref_counts[block_id] -= 1
        if self.ref_counts[block_id] == 0:
            self.free_blocks.append(block_id)

    def register_sequence(self, seq_id: int):
        """Register a new sequence with an initial block."""
        self.block_table[seq_id] = [self._alloc_block()]

    def append_token(self, seq_id: int, token_position: int) -> int:
        """
        Append a token to a sequence. Allocates a new block if needed.
        Returns the physical block ID for this token.
        """
        blocks = self.block_table[seq_id]
        slot_in_current_block = token_position % self.block_size
        # Need a new block when we've filled the current one
        if token_position > 0 and slot_in_current_block == 0:
            new_block = self._alloc_block()
            blocks.append(new_block)
        return blocks[-1]

    def lookup_token(self, seq_id: int, token_position: int) -> tuple:
        """Return (physical_block_id, slot_within_block) for a token position."""
        block_idx = token_position // self.block_size
        slot = token_position % self.block_size
        physical_block = self.block_table[seq_id][block_idx]
        return physical_block, slot

    def free_sequence(self, seq_id: int):
        """Release all blocks for a completed sequence."""
        for block_id in self.block_table[seq_id]:
            self._free_block(block_id)
        del self.block_table[seq_id]

    def fork_sequence(self, src_seq_id: int, dst_seq_id: int):
        """Copy-on-write fork for beam search: share all blocks."""
        src_blocks = self.block_table[src_seq_id]
        self.block_table[dst_seq_id] = src_blocks.copy()
        for block_id in src_blocks:
            self.ref_counts[block_id] += 1

    @property
    def num_free_blocks(self) -> int:
        return len(self.free_blocks)

    @property
    def memory_utilization(self) -> float:
        used = self.num_blocks - self.num_free_blocks
        return used / self.num_blocks if self.num_blocks > 0 else 0.0


# --- Demo ---
print("=== PagedKVCacheManager Demo ===")
print()

# Simulate LLaMA-7B config with 8 GB for KV cache
manager = PagedKVCacheManager(
    total_gpu_memory_gb=8.0,
    block_size=16,        # 16 tokens per block (common default in vLLM)
    n_layers=32,
    n_kv_heads=32,
    d_head=128,
    dtype_bytes=2,
)

bytes_per_block_kb = manager.bytes_per_block / 1024
print(f"Model: LLaMA-7B config")
print(f"Block size: {manager.block_size} tokens")
print(f"Memory per block: {bytes_per_block_kb:.1f} KB")
print(f"Total blocks available: {manager.num_blocks}")
print(f"Max tokens storable: {manager.num_blocks * manager.block_size:,}")
print()

# Register three sequences
for seq_id in [1, 2, 3]:
    manager.register_sequence(seq_id)

print(f"After registering 3 sequences: {manager.num_free_blocks} free blocks")

# Simulate generating 100 tokens for sequence 1
for pos in range(1, 100):
    manager.append_token(1, pos)

blocks_used_seq1 = len(manager.block_table[1])
print(f"Seq 1 (100 tokens) uses {blocks_used_seq1} blocks "
      f"(ceil(100/{manager.block_size}) = {math.ceil(100/manager.block_size)})")

# Fork sequence 1 for beam search
manager.fork_sequence(1, 4)
print(f"After forking seq 1 -> seq 4: blocks are shared (copy-on-write)")
print(f"  Seq 1 blocks: {manager.block_table[1][:3]}... (ref_count > 1)")
print(f"  Seq 4 blocks: {manager.block_table[4][:3]}... (same physical blocks)")

# Lookup token position
phys_block, slot = manager.lookup_token(1, 47)
print(f"\nToken at position 47 of seq 1: physical_block={phys_block}, slot={slot}")

# Free sequence 2
manager.free_sequence(2)
print(f"\nAfter freeing seq 2: {manager.num_free_blocks} free blocks returned")

# Naive contiguous comparison
print("\n=== Naive Contiguous Allocation Comparison ===")
max_seq_len = 2048
bytes_per_seq_contiguous = 2 * 32 * 32 * 128 * max_seq_len * 2
max_concurrent_naive = int(8 * (1024**3)) // bytes_per_seq_contiguous
max_concurrent_paged = manager.num_blocks * manager.block_size // 512  # avg 512 tokens
print(f"Naive (pre-allocate 2K tokens/seq): max {max_concurrent_naive} concurrent seqs")
print(f"PagedAttention (avg 512 tokens/seq): max ~{max_concurrent_paged} concurrent seqs")
print(f"Throughput improvement: ~{max_concurrent_paged / max(max_concurrent_naive,1):.1f}x")


=== PagedKVCacheManager Demo ===

Model: LLaMA-7B config
Block size: 16 tokens
Memory per block: 8192.0 KB
Total blocks available: 1024
Max tokens storable: 16,384

After registering 3 sequences: 1021 free blocks
Seq 1 (100 tokens) uses 7 blocks (ceil(100/16) = 7)
After forking seq 1 -> seq 4: blocks are shared (copy-on-write)
  Seq 1 blocks: [1023, 1020, 1019]... (ref_count > 1)
  Seq 4 blocks: [1023, 1020, 1019]... (same physical blocks)

Token at position 47 of seq 1: physical_block=1019, slot=15

After freeing seq 2: 1016 free blocks returned

=== Naive Contiguous Allocation Comparison ===
Naive (pre-allocate 2K tokens/seq): max 8 concurrent seqs
PagedAttention (avg 512 tokens/seq): max ~32 concurrent seqs
Throughput improvement: ~4.0x


## FlashAttention: Fused SRAM Kernel

### The Standard Attention Bottleneck

Standard attention computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

This requires materializing the full $N \times N$ attention matrix $S = QK^T / \sqrt{d_k}$ in GPU HBM (high-bandwidth memory). For sequence length $N = 8192$, this is $8192^2 \approx 67M$ float16 values ≈ **134 MB** per head per layer. With 32 heads and 32 layers, the attention matrices alone require **137 GB** impossible on any single GPU.

Even for shorter sequences, the HBM reads/writes dominate runtime. Standard attention performs:
- Write $S$ ($N^2$) to HBM
- Read $S$, write $P = \text{softmax}(S)$ to HBM
- Read $P$, $V$, write output to HBM

Total HBM accesses: $O(N^2)$ this is the bottleneck, not the FLOPs.

### FlashAttention: Tiling on SRAM

FlashAttention (Dao et al., 2022) avoids materializing the attention matrix by **tiling**: split $Q$, $K$, $V$ into blocks that fit in the fast on-chip SRAM (typically 20-40 MB on modern GPUs).

For each query block $Q_i$, iterate over all key/value blocks $(K_j, V_j)$:

1. Load $Q_i$, $K_j$, $V_j$ into SRAM
2. Compute $S_{ij} = Q_i K_j^T / \sqrt{d_k}$
3. Maintain **running softmax statistics** $(m_i, \ell_i)$: current max and sum of exponentials
4. Update the output accumulator with the rescaled contribution

The **online softmax trick** allows computing the correct softmax without seeing all elements first:

$$m_i^{\text{new}} = \max(m_i^{\text{old}}, \max_j S_{ij})$$
$$\ell_i^{\text{new}} = e^{m_i^{\text{old}} - m_i^{\text{new}}} \cdot \ell_i^{\text{old}} + \sum_j e^{S_{ij} - m_i^{\text{new}}}$$

The final output is rescaled: $O_i = O_i^{\text{accumulated}} / \ell_i^{\text{new}}$

### Complexity Comparison

| Metric | Standard Attention | FlashAttention v1 | FlashAttention v2 |
|---|---|---|---|
| HBM reads/writes | $O(N^2)$ | $O(N)$ | $O(N)$ |
| FLOP count | $O(N^2 d)$ | $O(N^2 d)$ | $O(N^2 d)$ |
| Memory | $O(N^2)$ | $O(N)$ | $O(N)$ |
| Parallelism | | batch, heads | + sequence dim |
| Speedup (A100) | 1× | 2-4× | 5-9× |

FlashAttention does **not** reduce FLOPs the same multiplications are performed. It reduces HBM bandwidth consumption, which is the actual bottleneck.

### Versions and Variants

- **FlashAttention v1** (2022): established the tiling approach; $O(N)$ memory; 2-4× speedup.
- **FlashAttention v2** (2023): better work partitioning across thread blocks; reduced non-matmul FLOPs; improved parallelism over the sequence dimension; 5-9× speedup over standard attention.
- **FlashAttention v3** (2024): targets Hopper (H100) architecture; FP8 support; warp specialization; overlaps compute and memory operations; reaches 1.5-2× speedup over FA2.
- **FlashDecoding** (2023): FlashAttention parallelizes over the query dimension but autoregressive decode has query length 1. FlashDecoding parallelizes over the KV sequence length instead, enabling efficient long-context generation on multiple GPUs/SMs.

### Integration

```python
# HuggingFace Transformers
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
)
```

FlashAttention is now the default in most production frameworks and is enabled automatically in vLLM, TGI, and TensorRT-LLM when the GPU architecture supports it (Ampere, Hopper).


## Speculative Decoding: Draft-Then-Verify

### The Sequential Bottleneck

Autoregressive generation is inherently sequential: token $t+1$ depends on token $t$. Each forward pass through a 70B parameter model generates exactly 1 token. A GPU capable of running a 70B model in ~20 ms/token cannot be sped up by simply asking it to "go faster" the memory bandwidth is saturated reading model weights.

**Key insight**: A GPU can process $K$ tokens in nearly the same time as 1 token, because the bottleneck is loading model weights (fixed cost), not the matrix multiplications (variable but small for small batch).

### Algorithm

Speculative decoding (Leviathan et al., 2022; Chen et al., 2023) uses a small **draft model** $q$ (e.g., 7B) and a large **target model** $p$ (e.g., 70B):

1. Draft model autoregressively generates $K$ candidate tokens $\tilde{x}_1, \ldots, \tilde{x}_K$
2. Target model processes the prefix + all $K$ draft tokens **in one forward pass** (parallel)
3. Accept/reject each draft token based on probability ratio:

$$\alpha_i = \min\left(1,\ \frac{p(\tilde{x}_i \mid x_{<i})}{q(\tilde{x}_i \mid x_{<i})}\right)$$

Accept token $\tilde{x}_i$ with probability $\alpha_i$. If accepted, move to $\tilde{x}_{i+1}$. If rejected, sample a **corrected** token from $\max(0, p - q)$ and stop.

The procedure is **lossless**: the output distribution is identical to sampling directly from $p$.

### Expected Speedup

Let $\alpha$ be the average acceptance probability. Expected number of tokens generated per speculative step:

$$\mathbb{E}[\#\text{tokens}] = \frac{1 - \alpha^{K+1}}{1 - \alpha}$$

If $\alpha = 0.9$ and $K = 5$, expected tokens per step $\approx 4.7$ nearly a 5× speedup over single-token generation, as long as the draft + verify cost is comparable to 1 large model step.

### Variants

| Variant | Draft Source | Key Idea |
|---|---|---|
| Classic Speculative | Separate small model | 2-model pipeline |
| Medusa | Extra heads on base model | Multiple parallel draft heads, no separate model |
| EAGLE | Autoregressive draft head | Feature-level speculation, higher acceptance |
| Lookahead Decoding | Jacobi iteration | No draft model; uses n-gram lookahead |
| Prompt Lookup | Input token n-grams | Copies matching substrings from prompt |

**Medusa** adds $K$ separate classifier heads to the base model, each predicting $k$ steps ahead. The draft is generated in a single base model forward pass, avoiding draft model memory overhead.

**EAGLE** trains a lightweight autoregressive head that operates on the feature space (embeddings) rather than token space, achieving higher acceptance rates (~80-90%) than classic speculative decoding.


In [3]:
import random
import math
from typing import List, Tuple


def sample_from_dist(probs: List[float]) -> int:
    """Sample an index from a probability distribution."""
    r = random.random()
    cumulative = 0.0
    for i, p in enumerate(probs):
        cumulative += p
        if r <= cumulative:
            return i
    return len(probs) - 1


def normalize(probs: List[float]) -> List[float]:
    total = sum(probs)
    return [p / total for p in probs] if total > 0 else probs


def speculative_decode(
    target_probs: List[float],   # p(token) from large target model
    draft_probs: List[float],    # q(token) from small draft model
    K: int = 5,
    seed: int = 42,
) -> Tuple[List[int], int, int]:
    """
    Toy speculative decoding over a single vocabulary step.
    Returns: (accepted_tokens, num_accepted, num_draft_calls)

    In a real system, target_probs and draft_probs would change
    at each position. Here we use fixed distributions for illustration.
    """
    random.seed(seed)
    vocab_size = len(target_probs)
    target_probs = normalize(target_probs)
    draft_probs = normalize(draft_probs)

    accepted_tokens = []
    total_target_calls = 1  # One batched call to verify K+1 positions

    # Step 1: Draft model generates K tokens autoregressively
    draft_tokens = [sample_from_dist(draft_probs) for _ in range(K)]

    # Step 2: Target model verifies all K tokens in one forward pass
    # (simulated: we already have target_probs)
    for k, x_tilde in enumerate(draft_tokens):
        p_x = target_probs[x_tilde]
        q_x = draft_probs[x_tilde]

        alpha = min(1.0, p_x / (q_x + 1e-10))
        u = random.random()

        if u <= alpha:
            # Accept draft token
            accepted_tokens.append(x_tilde)
        else:
            # Reject: sample corrected token from max(0, p - q)
            corrected = [max(0.0, target_probs[i] - draft_probs[i])
                         for i in range(vocab_size)]
            corrected = normalize(corrected)
            corrected_token = sample_from_dist(corrected)
            accepted_tokens.append(corrected_token)
            # Stop after rejection
            return accepted_tokens, len(accepted_tokens), K

    # All K accepted: sample one more from target at position K+1
    bonus_token = sample_from_dist(target_probs)
    accepted_tokens.append(bonus_token)
    return accepted_tokens, len(accepted_tokens), K


def expected_tokens_per_step(alpha: float, K: int) -> float:
    """Theoretical expected accepted tokens per speculative step."""
    if alpha >= 1.0:
        return K + 1.0
    return (1.0 - alpha ** (K + 1)) / (1.0 - alpha)


# --- Simulation ---
random.seed(0)
vocab_size = 32

# Target distribution: peaked around token 5
target = [math.exp(-0.5 * (i - 5) ** 2) for i in range(vocab_size)]
# Draft distribution: similar but slightly shifted simulate a weaker model
draft = [math.exp(-0.5 * (i - 6) ** 2) for i in range(vocab_size)]

print("=== Speculative Decoding Simulation ===")
print()

# Run 1000 trials
n_trials = 1000
K = 5
total_accepted = 0
for trial in range(n_trials):
    tokens, n_acc, _ = speculative_decode(target, draft, K=K, seed=trial)
    total_accepted += n_acc

empirical_avg = total_accepted / n_trials
print(f"K = {K} draft tokens per step")
print(f"Empirical avg tokens accepted per step: {empirical_avg:.3f}")

# Theoretical analysis for different alpha values
print()
print(f"{'alpha':>8} {'K=3':>10} {'K=5':>10} {'K=7':>10} {'Speedup(K=5)':>14}")
print("-" * 55)
for alpha in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99]:
    e3 = expected_tokens_per_step(alpha, 3)
    e5 = expected_tokens_per_step(alpha, 5)
    e7 = expected_tokens_per_step(alpha, 7)
    # Speedup: tokens_per_step / cost_ratio
    # Assume draft is 10x cheaper than target; each step costs 1 + K/10 target-equivalents
    draft_cost_ratio = 0.1
    cost_per_step = 1.0 + K * draft_cost_ratio  # in target model time units
    speedup = e5 / cost_per_step
    print(f"{alpha:>8.2f} {e3:>10.3f} {e5:>10.3f} {e7:>10.3f} {speedup:>14.2f}x")

print()
print("Speedup assumes draft model is 10x cheaper than target model.")

# Example run with output
print()
print("=== Example Speculative Decode Run ===")
tokens, n_acc, n_draft = speculative_decode(target, draft, K=5, seed=7)
print(f"Draft tokens generated: {n_draft}")
print(f"Tokens accepted: {n_acc} (tokens: {tokens})")
print(f"Tokens per target call: {n_acc} (vs 1 for standard)")


=== Speculative Decoding Simulation ===



K = 5 draft tokens per step
Empirical avg tokens accepted per step: 2.368

   alpha        K=3        K=5        K=7   Speedup(K=5)
-------------------------------------------------------
    0.50      1.875      1.969      1.992           1.31x
    0.60      2.176      2.383      2.458           1.59x
    0.70      2.533      2.941      3.141           1.96x
    0.80      2.952      3.689      4.161           2.46x
    0.85      3.187      4.152      4.850           2.77x
    0.90      3.439      4.686      5.695           3.12x
    0.95      3.710      5.298      6.732           3.53x
    0.99      3.940      5.852      7.726           3.90x

Speedup assumes draft model is 10x cheaper than target model.



=== Example Speculative Decode Run ===
Draft tokens generated: 5
Tokens accepted: 6 (tokens: [6, 5, 6, 5, 6, 4])
Tokens per target call: 6 (vs 1 for standard)


## Prompt Caching and Prefix Caching

### The Repeated Prefix Problem

Many LLM applications repeatedly process the same prefix:
- **System prompts**: "You are a helpful assistant. Follow these rules: ..." (often thousands of tokens)
- **Few-shot examples**: The same 5-10 examples prepended to every query
- **Long documents**: RAG pipelines that prepend the same context to multiple questions
- **Multi-turn conversations**: Prior turns are re-processed with each new message

Without caching, every request pays the full prefill cost for the repeated prefix wasting compute and increasing latency.

### Anthropic Prompt Caching

The Anthropic API supports explicit **cache breakpoints**: the user marks specific prefix positions as cacheable using `cache_control: {"type": "ephemeral"}`. The KV cache for those tokens is stored server-side for up to 5 minutes (Claude 3.5+ models).

**Cost impact**: Cached tokens cost ~10% of normal input token pricing, and retrieval is included in the API response (`cache_read_input_tokens`). For applications with large stable prefixes (documents, system prompts), this yields dramatic latency and cost reductions.

```python
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": long_document_text,
                "cache_control": {"type": "ephemeral"},  # Mark as cacheable
            },
            {"type": "text", "text": user_question},
        ],
    }
]
```

### vLLM Automatic Prefix Caching (APC)

vLLM implements **automatic prefix caching** without requiring explicit user annotations:

1. When a request arrives, its prompt tokens are hashed block-by-block.
2. The block manager checks if a physical block matching each prefix hash already exists.
3. Matching blocks are **reused** directly (no recomputation); only novel suffix blocks are computed.
4. The block eviction policy (LRU or similar) manages the cache when memory pressure arises.

This requires no changes to the client API. Any two requests sharing a common prefix will automatically benefit.

### Savings Analysis

| Scenario | Prefix Fraction | Latency Reduction | Throughput Gain |
|---|---|---|---|
| Long system prompt (2K/4K total) | 50% | ~40% | ~2× |
| Document QA (4K doc / 4.1K total) | ~98% | ~95% | ~15× |
| Few-shot (1K/2K total) | 50% | ~40% | ~1.5× |
| Multi-turn (growing) | ~70-90% | ~60-80% | ~3-8× |

### SGLang RadixAttention

SGLang extends prefix caching with **RadixAttention**: a radix tree structure that enables sharing at arbitrary positions, not just exact prefix matches. This allows multi-program, multi-query workflows to share intermediate KV states from branching LLM programs.


## Quantization Formats Deep Dive

### Precision Overview

| Format | Bits | Range | Precision Notes | Typical Use |
|---|---|---|---|---|
| FP32 | 32 | ±3.4×10³⁸ | Full precision | Training, CPU |
| FP16 | 16 | ±65504 | Limited range, overflow risk | GPU inference |
| BF16 | 16 | ±3.4×10³⁸ | Same range as FP32, less mantissa | Modern GPU training/inference |
| INT8 | 8 | −128-127 | Integer; needs scale/zero-point | W8A8, LLM.int8() |
| INT4 | 4 | −8-7 | Very coarse; group quantization | GPTQ, AWQ, GGUF |
| FP8 (E4M3) | 8 | ±448 | Float 8-bit; H100 native | Training, TensorRT-LLM |
| NF4 | 4 | Normalized float | QLoRA NormalFloat | QLoRA fine-tuning |

### LLM.int8() Mixed Precision INT8

Bitsandbytes (Dettmers et al., 2022) discovered that LLM activations contain **systematic outliers**: a small fraction (~0.1%) of feature dimensions have values 100× larger than the rest. Naive INT8 quantization would need to scale to accommodate these outliers, destroying precision for normal values.

**Solution**: Decompose the matrix multiplication:
- Identify outlier columns in activations (threshold: $|x| > 6.0$)
- Compute the outlier portion in FP16
- Quantize the remaining 99.9% of values to INT8
- Add the two results

This gives near-zero accuracy degradation at ~2× memory reduction.

### GPTQ Post-Training Layer-Wise Quantization

GPTQ (Frantar et al., 2022) quantizes weights by minimizing the layer-wise output error:

$$\min_{\hat{W}} \|WX - \hat{W}X\|_F^2$$

where $X$ is a small calibration dataset. Uses the **Optimal Brain Quantization (OBQ)** framework:
- Compute the Hessian $H = XX^T$
- Quantize one weight at a time
- Update remaining weights to compensate: $\delta W_j = -\frac{q(W_i) - W_i}{H_{ii}^{-1}} \cdot H_{ij}^{-1}$

GPTQ achieves INT4 quantization with <1% perplexity increase. Block-wise column ordering and lazy batch updates make it practical on large models.

### AWQ Activation-Aware Weight Quantization

AWQ (Lin et al., 2023) identifies **salient weights** by examining activation magnitudes. If an activation channel $x_c$ has high average magnitude, the corresponding weight row $W_c$ is more important.

**Key insight**: Protect the top 1% of weight channels (by activation magnitude) by scaling them before quantization:

$$\hat{W}_c = W_c \cdot s_c, \quad \hat{x}_c = x_c / s_c$$

The scale $s_c$ is chosen to minimize quantization error for the important channels. This costs no extra memory (scales are absorbed into LayerNorm) and achieves better accuracy than GPTQ, especially at INT4.

### SmoothQuant Activation Quantization

SmoothQuant (Xiao et al., 2022) enables **W8A8** (both weights and activations INT8) inference. The problem: activations are hard to quantize (large outliers), while weights are easy.

**Migrate difficulty from activations to weights**:

$$Y = (X \cdot \text{diag}(s)^{-1}) \cdot (\text{diag}(s) \cdot W) = \hat{X} \cdot \hat{W}$$

where $s_c = \max(|X_c|)^\alpha / \max(|W_c|)^{1-\alpha}$ and $\alpha \in [0, 1]$ controls the migration strength. Both $\hat{X}$ and $\hat{W}$ can now be quantized to INT8, enabling 2× speedup from INT8 tensor cores with minimal accuracy loss.

### GGUF/GGML CPU-Friendly Mixed Precision

GGUF (llama.cpp format) defines a family of quantization formats optimized for CPU inference:
- **Q4_K_M**: 4-bit quantization with K-means quantization, medium accuracy
- **Q5_K_S**: 5-bit, small variant, better accuracy at 25% more memory
- **Q8_0**: 8-bit baseline, near-lossless on CPU
- Each format defines group size (typically 32 or 64 weights share a scale)

### Other Notable Formats

| Format | Key Idea |
|---|---|
| **SpQR** | Sparse + quantized; outlier weights kept in FP16 sparse format |
| **AQLM** | Additive quantization; residual codebook decomposition |
| **QuIP** | Incoherence processing; random rotation before INT4 for near-lossless |
| **HQQ** | Half-quadratic quantization; fast, no calibration data needed |
| **EETQ** | INT8 for weights, FP16 for activations; fast CUDA kernels; easy drop-in |


In [4]:
CODE_GPTQ = '''
# GPTQ Quantization with auto-gptq
# pip install auto-gptq optimum

from transformers import AutoTokenizer
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

model_id = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Calibration dataset (small, ~128 samples is enough)
calibration_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming the world.",
    # ... more samples ...
]
calibration_data = [
    tokenizer(text, return_tensors="pt").input_ids
    for text in calibration_texts
]

quantize_config = BaseQuantizeConfig(
    bits=4,           # Quantize to INT4
    group_size=128,   # 128 weights share one scale factor
    desc_act=False,   # Don't use activation ordering (faster quantization)
)

model = AutoGPTQForCausalLM.from_pretrained(
    model_id,
    quantize_config=quantize_config,
)

# Run GPTQ calibration and quantization
model.quantize(calibration_data)

# Save quantized model
model.save_quantized("llama-2-7b-gptq-int4", use_safetensors=True)
tokenizer.save_pretrained("llama-2-7b-gptq-int4")

# Load and run inference
model_q = AutoGPTQForCausalLM.from_quantized(
    "llama-2-7b-gptq-int4",
    use_safetensors=True,
    device="cuda:0",
)
inputs = tokenizer("Hello, I am", return_tensors="pt").to("cuda:0")
output = model_q.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))
'''
print(CODE_GPTQ)



# GPTQ Quantization with auto-gptq
# pip install auto-gptq optimum

from transformers import AutoTokenizer
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

model_id = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Calibration dataset (small, ~128 samples is enough)
calibration_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming the world.",
    # ... more samples ...
]
calibration_data = [
    tokenizer(text, return_tensors="pt").input_ids
    for text in calibration_texts
]

quantize_config = BaseQuantizeConfig(
    bits=4,           # Quantize to INT4
    group_size=128,   # 128 weights share one scale factor
    desc_act=False,   # Don't use activation ordering (faster quantization)
)

model = AutoGPTQForCausalLM.from_pretrained(
    model_id,
    quantize_config=quantize_config,
)

# Run GPTQ calibration and quantization
model.quantize(calibration_data)

# Save quantized model
m

In [5]:
CODE_AWQ = '''
# AWQ Quantization with autoawq
# pip install autoawq

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_id = "meta-llama/Llama-2-7b-hf"
quant_path = "llama-2-7b-awq-int4"

quant_config = {
    "zero_point": True,   # Use zero-point quantization
    "q_group_size": 128,  # Group size for quantization
    "w_bit": 4,           # 4-bit weights
    "version": "GEMM",    # Use GEMM kernel (faster for large batches)
    # "version": "GEMV",  # Use GEMV kernel (faster for batch=1)
}

model = AutoAWQForCausalLM.from_pretrained(model_id, safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# AWQ uses activation statistics to find salient weights
# Calibration happens automatically with a small dataset
model.quantize(tokenizer, quant_config=quant_config)

# Save
model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)

# Load quantized model for inference
model_q = AutoAWQForCausalLM.from_quantized(
    quant_path,
    fuse_layers=True,    # Fuse operations for speed
)

# Generate text
inputs = tokenizer("The future of AI is", return_tensors="pt").to("cuda")
output = model_q.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
print(tokenizer.decode(output[0], skip_special_tokens=True))

# Memory comparison
# FP16 (7B params):  ~14 GB
# AWQ INT4 (7B):      ~4 GB (3.5x reduction)
'''
print(CODE_AWQ)



# AWQ Quantization with autoawq
# pip install autoawq

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_id = "meta-llama/Llama-2-7b-hf"
quant_path = "llama-2-7b-awq-int4"

quant_config = {
    "zero_point": True,   # Use zero-point quantization
    "q_group_size": 128,  # Group size for quantization
    "w_bit": 4,           # 4-bit weights
    "version": "GEMM",    # Use GEMM kernel (faster for large batches)
    # "version": "GEMV",  # Use GEMV kernel (faster for batch=1)
}

model = AutoAWQForCausalLM.from_pretrained(model_id, safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# AWQ uses activation statistics to find salient weights
# Calibration happens automatically with a small dataset
model.quantize(tokenizer, quant_config=quant_config)

# Save
model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)

# Load quantized model for inference
model_q = AutoAWQForCausalLM.from_quantized(
    quant_path,
    fuse_layers=

In [6]:
CODE_GGUF = '''
# Loading and running GGUF models with llama-cpp-python
# pip install llama-cpp-python
# For GPU support: CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python

from llama_cpp import Llama

# Download a GGUF model from HuggingFace (e.g., TheBloke or bartowski repos)
# Example: Llama-2-7B-Chat Q4_K_M quantization

# Option 1: Load from local file
llm = Llama(
    model_path="./llama-2-7b-chat.Q4_K_M.gguf",
    n_ctx=4096,       # Context window size
    n_gpu_layers=35,  # Number of layers to offload to GPU (0 = CPU only)
    n_threads=8,      # CPU threads for remaining layers
    verbose=False,
)

# Option 2: Download directly from HuggingFace
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(
    repo_id="TheBloke/Llama-2-7B-Chat-GGUF",
    filename="llama-2-7b-chat.Q4_K_M.gguf",
)
llm = Llama(model_path=model_path, n_ctx=4096, n_gpu_layers=35)

# Text completion
output = llm(
    "Question: What is the capital of France? Answer:",
    max_tokens=50,
    stop=["\n", "Question:"],
    echo=False,
)
print(output["choices"][0]["text"])

# Chat completion (OpenAI-compatible)
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain quantum entanglement in simple terms."},
    ],
    temperature=0.7,
    max_tokens=200,
)
print(response["choices"][0]["message"]["content"])

# Streaming output
stream = llm(
    "Once upon a time",
    max_tokens=100,
    stream=True,
)
for chunk in stream:
    token = chunk["choices"][0]["text"]
    print(token, end="", flush=True)
print()

# GGUF quantization formats comparison:
# Q4_K_S  : 4-bit, small  ~3.5 GB for 7B, fastest
# Q4_K_M  : 4-bit, medium ~4.0 GB for 7B, recommended balance
# Q5_K_M  : 5-bit, medium ~4.8 GB for 7B, better quality
# Q8_0    : 8-bit          ~7.0 GB for 7B, near-lossless
'''
print(CODE_GGUF)



# Loading and running GGUF models with llama-cpp-python
# pip install llama-cpp-python
# For GPU support: CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python

from llama_cpp import Llama

# Download a GGUF model from HuggingFace (e.g., TheBloke or bartowski repos)
# Example: Llama-2-7B-Chat Q4_K_M quantization

# Option 1: Load from local file
llm = Llama(
    model_path="./llama-2-7b-chat.Q4_K_M.gguf",
    n_ctx=4096,       # Context window size
    n_gpu_layers=35,  # Number of layers to offload to GPU (0 = CPU only)
    n_threads=8,      # CPU threads for remaining layers
    verbose=False,
)

# Option 2: Download directly from HuggingFace
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(
    repo_id="TheBloke/Llama-2-7B-Chat-GGUF",
    filename="llama-2-7b-chat.Q4_K_M.gguf",
)
llm = Llama(model_path=model_path, n_ctx=4096, n_gpu_layers=35)

# Text completion
output = llm(
    "Question: What is the capital of France? Answer:",
    max_tokens=50,
    s

## Inference Serving Frameworks

### Framework Comparison

| Framework | Key Feature | Best For | License |
|---|---|---|---|
| **vLLM** | PagedAttention, continuous batching, async engine | High-throughput API serving | Apache 2.0 |
| **TGI** (HuggingFace) | Tensor parallelism, Rust core, HF native | HuggingFace model serving | Apache 2.0 |
| **llama.cpp** | GGUF, CPU+Metal+CUDA, minimal deps | Consumer hardware, edge | MIT |
| **Ollama** | Model management, REST API, desktop | Developer experimentation | MIT |
| **TensorRT-LLM** | NVIDIA-optimized, FP8, custom kernels | Maximum GPU throughput | NVIDIA |
| **Triton Inference Server** | Multi-model, multiple backends, MLOps | Production multi-model serving | BSD |
| **SGLang** | Structured generation, RadixAttention | Complex LLM programs | Apache 2.0 |

### vLLM

vLLM is the most widely adopted open-source LLM serving framework. Core features:
- **PagedAttention**: near-zero KV cache fragmentation, described above
- **Continuous batching**: iteration-level scheduling (Orca-style)
- **Async LLM Engine**: non-blocking request processing
- **OpenAI-compatible API**: drop-in replacement for OpenAI endpoints
- **Tensor/Pipeline Parallelism**: multi-GPU via Megatron-style sharding
- **Quantization support**: GPTQ, AWQ, SqueezeLLM, bitsandbytes
- **Speculative decoding**: draft-model integration

### Text Generation Inference (TGI)

HuggingFace's production serving library:
- **Rust backend** for high-performance HTTP serving
- **Tensor parallelism** natively (multi-GPU, single-node)
- **Continuous batching** similar to vLLM
- **Paged Attention** (added post-vLLM)
- **Optimized for HF Hub models**: seamless integration, model cards, private repos
- **Streaming tokens**: Server-Sent Events for real-time token delivery

### llama.cpp

CPU-first inference engine supporting consumer hardware:
- Pure C/C++ with optional CUDA, Metal (Apple Silicon), SYCL backends
- GGUF format with mixed-precision quantization (Q4, Q5, Q8 per layer)
- Supports offloading some layers to GPU while computing others on CPU
- Works on Raspberry Pi, MacBook M-series, old GPUs, and servers
- Powers Ollama and many desktop apps

### Ollama

User-friendly wrapper around llama.cpp for developers:
- `ollama run llama3` one-command model download and run
- Automatic hardware detection and GPU offloading
- OpenAI-compatible REST API (`http://localhost:11434/v1`)
- Model library with versioned tags (Modelfile for customization)
- Cross-platform: macOS, Windows, Linux

### TensorRT-LLM

NVIDIA's highest-performance inference engine:
- Compiles models to optimized TensorRT engines
- **FP8 support** on H100 (Hopper architecture)
- Custom CUDA kernels for attention, activation, and normalization
- **In-flight batching** (continuous batching)
- **KV cache manager** similar to PagedAttention
- Best throughput/latency on NVIDIA hardware; 2-5× over HF baseline
- Complex build process; less flexible than vLLM for custom models

### NVIDIA Triton Inference Server

Enterprise multi-model serving platform:
- Supports **multiple backends**: TensorRT, PyTorch, TensorFlow, ONNX, custom Python
- **Dynamic batching** across requests
- **Model ensembles**: chain multiple models in a pipeline
- **gRPC and HTTP endpoints** with Prometheus metrics
- **Model repositories**: version management and A/B testing
- Pairs with TensorRT-LLM backend for LLM serving


In [7]:
CODE_VLLM = '''
# vLLM Server and Python API
# pip install vllm

# ============================================================
# 1. Start vLLM Server (command line)
# ============================================================
# Basic server:
#   python -m vllm.entrypoints.openai.api_server \
#       --model meta-llama/Llama-2-7b-chat-hf \
#       --port 8000
#
# With quantization:
#   python -m vllm.entrypoints.openai.api_server \
#       --model TheBloke/Llama-2-7B-Chat-AWQ \
#       --quantization awq \
#       --dtype float16 \
#       --port 8000
#
# Multi-GPU tensor parallel:
#   python -m vllm.entrypoints.openai.api_server \
#       --model meta-llama/Llama-2-70b-chat-hf \
#       --tensor-parallel-size 4 \
#       --port 8000

# ============================================================
# 2. Python Offline Inference (most common for batching)
# ============================================================
from vllm import LLM, SamplingParams

# Initialize engine
llm = LLM(
    model="meta-llama/Llama-2-7b-chat-hf",
    tensor_parallel_size=1,     # Number of GPUs
    gpu_memory_utilization=0.9, # Fraction of GPU memory for KV cache
    max_model_len=4096,         # Maximum sequence length
    quantization=None,          # "awq", "gptq", "squeezellm", etc.
    enable_prefix_caching=True, # Enable automatic prefix caching
)

# Define sampling parameters
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.95,
    max_tokens=512,
    stop=["</s>", "[INST]"],
)

# Batch inference: all prompts processed together with continuous batching
prompts = [
    "[INST] What is machine learning? [/INST]",
    "[INST] Explain neural networks briefly. [/INST]",
    "[INST] What is the transformer architecture? [/INST]",
    "[INST] How does backpropagation work? [/INST]",
]

outputs = llm.generate(prompts, sampling_params)

for i, output in enumerate(outputs):
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt[:50]}...")
    print(f"Output: {generated_text[:100]}...")
    print()

# ============================================================
# 3. Async Engine for Real-Time Serving
# ============================================================
import asyncio
from vllm import AsyncLLMEngine, AsyncEngineArgs

engine_args = AsyncEngineArgs(
    model="meta-llama/Llama-2-7b-chat-hf",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.9,
    max_model_len=4096,
)
engine = AsyncLLMEngine.from_engine_args(engine_args)

async def generate_streaming(prompt: str, request_id: str):
    """Stream tokens as they are generated."""
    sampling_params = SamplingParams(temperature=0.8, max_tokens=200)

    async for request_output in engine.generate(prompt, sampling_params, request_id):
        if request_output.outputs:
            token = request_output.outputs[0].text
            print(token, end="", flush=True)
    print()

# asyncio.run(generate_streaming("Explain LLMs in one paragraph:", "req-001"))

# ============================================================
# 4. OpenAI-Compatible Client (after starting server)
# ============================================================
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",
)

# Streaming chat completion
stream = client.chat.completions.create(
    model="meta-llama/Llama-2-7b-chat-hf",
    messages=[{"role": "user", "content": "What is vLLM?"}],
    stream=True,
    max_tokens=200,
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
'''
print(CODE_VLLM)



# vLLM Server and Python API
# pip install vllm

# ============================================================
# 1. Start vLLM Server (command line)
# ============================================================
# Basic server:
#   python -m vllm.entrypoints.openai.api_server #       --model meta-llama/Llama-2-7b-chat-hf #       --port 8000
#
# With quantization:
#   python -m vllm.entrypoints.openai.api_server #       --model TheBloke/Llama-2-7B-Chat-AWQ #       --quantization awq #       --dtype float16 #       --port 8000
#
# Multi-GPU tensor parallel:
#   python -m vllm.entrypoints.openai.api_server #       --model meta-llama/Llama-2-70b-chat-hf #       --tensor-parallel-size 4 #       --port 8000

# ============================================================
# 2. Python Offline Inference (most common for batching)
# ============================================================
from vllm import LLM, SamplingParams

# Initialize engine
llm = LLM(
    model="meta-llama/Llama-2-7b-c

## xFormers and Memory-Efficient Attention

### xFormers Library

xFormers (Meta AI) is a library of composable transformer building blocks with a focus on efficiency. Its attention module implements several variants:

- **Memory-efficient attention**: CUDA custom op that avoids materializing the full $N \times N$ attention matrix. Like FlashAttention, it uses tiling but with a different implementation approach that supports causal, cross-attention, and custom attention biases.
- **Blocked sparse attention**: for structured sparsity patterns
- **Variable-length sequences**: process sequences of different lengths without padding
- **Fused operations**: LayerNorm + dropout + residual in single kernel

### HuggingFace Integration

HuggingFace Transformers supports several attention backends via `attn_implementation`:

```python
from transformers import AutoModelForCausalLM
import torch

# FlashAttention 2 (requires flash-attn package, Ampere+ GPU)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
)

# PyTorch SDPA (scaled dot-product attention, uses best available kernel)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    attn_implementation="sdpa",  # automatic dispatch
    torch_dtype=torch.bfloat16,
)
```

PyTorch's `F.scaled_dot_product_attention` (SDPA, since PyTorch 2.0) dispatches to the best available kernel:
1. FlashAttention if available and inputs are compatible
2. Memory-efficient attention (xFormers kernel)
3. Math implementation (standard)

### Benchmark Comparison

Approximate benchmarks on A100 80GB, LLaMA-7B, batch=1, FP16:

| Attention Backend | Seq 512 (ms) | Seq 2K (ms) | Seq 8K (ms) | Peak Memory |
|---|---|---|---|---|
| Standard (eager) | 8.1 | 14.3 | OOM | $O(N^2)$ |
| PyTorch SDPA | 5.2 | 9.1 | 31.4 | $O(N)$ |
| xFormers mem-eff | 4.8 | 8.6 | 28.9 | $O(N)$ |
| FlashAttention v2 | 3.9 | 6.8 | 22.1 | $O(N)$ |
| FlashAttention v3 | 3.1 | 5.4 | 17.8 | $O(N)$ |

*Numbers are approximate and hardware/batch-size dependent.*

### When to Use What

| Scenario | Recommendation |
|---|---|
| Training on A100/H100 | FlashAttention v2/v3 |
| Training on older GPUs | xFormers mem-efficient |
| Inference with HF Transformers | `attn_implementation="sdpa"` (safe default) |
| vLLM / TGI serving | FlashAttention built-in |
| CPU inference | llama.cpp (no CUDA deps) |
| Custom attention patterns | xFormers (supports custom biases) |

### Torch Compile Integration

`torch.compile` (PyTorch 2.0+) can further optimize attention and other operations by fusing them into efficient Triton kernels. Combining `torch.compile` with FlashAttention often yields 10-30% additional speedup over FlashAttention alone, as the surrounding operations (projections, layer norms) are also fused.

```python
model = torch.compile(model, mode="reduce-overhead")  # or "max-autotune"
```


## Additional Learning Resources

### Foundational Papers

| Paper | ArXiv | Key Contribution |
|---|---|---|
| FlashAttention | [2205.14135](https://arxiv.org/abs/2205.14135) | IO-aware exact attention with SRAM tiling |
| FlashAttention-2 | [2307.08691](https://arxiv.org/abs/2307.08691) | Better parallelism, 5-9x over standard |
| FlashAttention-3 | [2407.08608](https://arxiv.org/abs/2407.08608) | Hopper/H100, FP8, warp specialization |
| PagedAttention / vLLM | [2309.06180](https://arxiv.org/abs/2309.06180) | Virtual memory for KV cache |
| Speculative Decoding (DeepMind) | [2302.01318](https://arxiv.org/abs/2302.01318) | Draft-verify with lossless distribution |
| Medusa | [2401.10774](https://arxiv.org/abs/2401.10774) | Multiple draft heads on base model |
| EAGLE | [2401.15077](https://arxiv.org/abs/2401.15077) | Feature-level autoregressive speculation |
| GPTQ | [2210.17323](https://arxiv.org/abs/2210.17323) | Post-training INT4 via OBQ |
| AWQ | [2306.00978](https://arxiv.org/abs/2306.00978) | Activation-aware INT4 quantization |
| SmoothQuant | [2211.10438](https://arxiv.org/abs/2211.10438) | W8A8 via activation-weight migration |
| Orca (Continuous Batching) | [2207.04236](https://arxiv.org/abs/2207.04236) | Iteration-level scheduling |
| FlashDecoding | [blog](https://crfm.stanford.edu/2023/10/12/flashdecoding.html) | Parallel decoding over KV sequence length |
| LLM.int8() | [2208.07339](https://arxiv.org/abs/2208.07339) | Mixed-precision INT8 with outlier decomposition |
| QuIP | [2307.13304](https://arxiv.org/abs/2307.13304) | Incoherence processing for near-lossless INT4 |

### Tools and Documentation

- **vLLM**: https://docs.vllm.ai installation, serving, quantization, performance tuning
- **llama.cpp GitHub**: https://github.com/ggerganov/llama.cpp GGUF formats, build instructions, Metal/CUDA
- **TGI (Text Generation Inference)**: https://huggingface.co/docs/text-generation-inference
- **Ollama**: https://ollama.com/docs model library, Modelfile, REST API reference
- **auto-gptq**: https://github.com/PanQiWei/AutoGPTQ GPTQ quantization for HuggingFace models
- **AutoAWQ**: https://github.com/casper-hansen/AutoAWQ AWQ quantization
- **bitsandbytes**: https://github.com/bitsandbytes-foundation/bitsandbytes LLM.int8(), NF4, QLoRA
- **FlashAttention GitHub**: https://github.com/Dao-AILab/flash-attention
- **xFormers**: https://github.com/facebookresearch/xformers
- **TensorRT-LLM**: https://github.com/NVIDIA/TensorRT-LLM
- **SGLang**: https://github.com/sgl-project/sglang structured generation, RadixAttention
- **HuggingFace Optimum**: https://huggingface.co/docs/optimum hardware-specific optimization (Intel, NVIDIA, AWS)

### Courses and Tutorials

- **MIT 6.5940: TinyML and Efficient Deep Learning** https://efficientml.ai quantization, pruning, KV cache, attention optimization (free lectures on YouTube)
- **HuggingFace LLM Course** https://huggingface.co/learn/llm-course practical inference, quantization, PEFT
- **Andrej Karpathy: LLM Inference** various YouTube talks and blog posts on practical LLM efficiency
- **Tim Dettmers Blog** https://timdettmers.com deep dives on quantization, bitsandbytes, LLM hardware selection
- **Lilian Weng: Large Transformer Model Inference Optimization** https://lilianweng.github.io/posts/2023-01-10-inference-optimization/ comprehensive survey blog post
